In [ ]:
# =========================================================
# H&M Customer Segmentation
# Portfolio Reconstruction Version
# Based on the original report / PPT workflow
# =========================================================

# 1. Import Libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ---------------------------------------------------------
# 2. Load Data
# ---------------------------------------------------------
customers = pd.read_csv("../data/customers.csv")
transactions = pd.read_csv("../data/transactions_train.csv")

print("customers shape:", customers.shape)
print("transactions shape:", transactions.shape)

# ---------------------------------------------------------
# 3. Customer Preprocessing
#    - Drop FN, Active, postal_code
#    - Drop missing rows
#    - Remove rare category 'None' in fashion_news_frequency
#    - Remove age outliers (> 86.5)
#    - Create age_group using Q3
# ---------------------------------------------------------
df_cus = customers.copy()

drop_cols = [col for col in ["FN", "Active", "postal_code"] if col in df_cus.columns]
df_cus = df_cus.drop(columns=drop_cols)

print("\n[Before missing value drop]")
print(df_cus.isnull().sum())

# 결측치 행 삭제
df_cus = df_cus.dropna().copy()

# fashion_news_frequency의 'None' 2건 제거
if "fashion_news_frequency" in df_cus.columns:
    df_cus = df_cus[df_cus["fashion_news_frequency"] != "None"].copy()

# age 이상치 제거 (보고서 기준 86.5세 이상 제거)
if "age" in df_cus.columns:
    df_cus = df_cus[df_cus["age"] <= 86.5].copy()

# Q3 기준 age_group 생성
q3_age = df_cus["age"].quantile(0.75)

df_cus["age_group"] = np.where(
    df_cus["age"] <= q3_age,
    "Target customer",
    "Non-target customer"
)

print("\n[After customer preprocessing]")
print(df_cus.shape)
print(df_cus.head())

# ---------------------------------------------------------
# 4. Transaction Preprocessing
#    - Convert date
#    - Create customer-level transaction count
#    - Define preferred channel as first purchase channel
# ---------------------------------------------------------
df_tr = transactions.copy()

df_tr["t_dat"] = pd.to_datetime(df_tr["t_dat"])

# 고객별 거래횟수
tr_count = (
    df_tr.groupby("customer_id")
         .size()
         .reset_index(name="Transactions")
)

# 고객별 첫 구매 채널
first_channel = (
    df_tr.sort_values(["customer_id", "t_dat"])
         .groupby("customer_id", as_index=False)
         .first()[["customer_id", "sales_channel_id"]]
)

# 채널 라벨 변환: PPT 기준 1=online, 2=offline
channel_map = {1: "online", 2: "offline"}
first_channel["preferred_channel"] = first_channel["sales_channel_id"].map(channel_map)

first_channel = first_channel[["customer_id", "preferred_channel"]]

print("\n[Transaction features]")
print(tr_count.head())
print(first_channel.head())

# ---------------------------------------------------------
# 5. Integration
#    - Merge customer + transaction features
# ---------------------------------------------------------
df = df_cus.merge(tr_count, on="customer_id", how="left")
df = df.merge(first_channel, on="customer_id", how="left")

# 거래 이력이 없는 고객 처리
df["Transactions"] = df["Transactions"].fillna(0)
df["preferred_channel"] = df["preferred_channel"].fillna("unknown")

print("\n[Integrated dataframe]")
print(df.shape)
print(df.head())

# ---------------------------------------------------------
# 6. Select Features for Clustering
#    Original project logic:
#    age, Transactions, club_member_status,
#    fashion_news_frequency, age_group, preferred_channel
# ---------------------------------------------------------
feature_cols = [
    "age",
    "Transactions",
    "club_member_status",
    "fashion_news_frequency",
    "age_group",
    "preferred_channel"
]

cluster_df = df[feature_cols].copy()

# 범주형 변수 one-hot encoding
cluster_df = pd.get_dummies(
    cluster_df,
    columns=[
        "club_member_status",
        "fashion_news_frequency",
        "age_group",
        "preferred_channel"
    ],
    drop_first=False
)

print("\n[Encoded clustering dataframe]")
print(cluster_df.shape)
print(cluster_df.head())

# ---------------------------------------------------------
# 7. Scaling
# ---------------------------------------------------------
scaler = StandardScaler()
X = scaler.fit_transform(cluster_df)

# ---------------------------------------------------------
# 8. Compare K-means and GMM (k = 2 ~ 10)
#    - The original project selected 7 by interpretability
#    - Here we additionally compute silhouette for portfolio quality
# ---------------------------------------------------------
results = []

for k in range(2, 11):
    # K-means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(X)
    kmeans_sil = silhouette_score(X, kmeans_labels)

    # GMM
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm_labels = gmm.fit_predict(X)
    gmm_sil = silhouette_score(X, gmm_labels)
    gmm_bic = gmm.bic(X)

    results.append({
        "k": k,
        "kmeans_silhouette": kmeans_sil,
        "gmm_silhouette": gmm_sil,
        "gmm_bic": gmm_bic
    })

comparison_df = pd.DataFrame(results)
print("\n[Model comparison]")
print(comparison_df)

# ---------------------------------------------------------
# 9. Final Clustering
#    Original project final choice: GMM with 7 clusters
# ---------------------------------------------------------
final_k = 7

kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
df["kmeans_cluster"] = kmeans_final.fit_predict(X)

gmm_final = GaussianMixture(n_components=final_k, random_state=42)
df["gmm_cluster"] = gmm_final.fit_predict(X)

print("\n[K-means cluster counts]")
print(df["kmeans_cluster"].value_counts().sort_index())

print("\n[GMM cluster counts]")
print(df["gmm_cluster"].value_counts().sort_index())

# ---------------------------------------------------------
# 10. Cluster Profiling (GMM)
# ---------------------------------------------------------
profile_num = (
    df.groupby("gmm_cluster")[["age", "Transactions"]]
      .agg(["mean", "median", "count"])
      .round(2)
)
print("\n[Numeric profile]")
print(profile_num)

# 각 군집의 대표 범주 확인 함수
def top_category_by_cluster(data, cluster_col, target_col):
    temp = (
        data.groupby([cluster_col, target_col])
            .size()
            .reset_index(name="count")
            .sort_values([cluster_col, "count"], ascending=[True, False])
    )
    top_temp = temp.groupby(cluster_col).head(1).reset_index(drop=True)
    return top_temp

top_status = top_category_by_cluster(df, "gmm_cluster", "club_member_status")
top_news = top_category_by_cluster(df, "gmm_cluster", "fashion_news_frequency")
top_channel = top_category_by_cluster(df, "gmm_cluster", "preferred_channel")
top_age_group = top_category_by_cluster(df, "gmm_cluster", "age_group")

print("\n[Top club_member_status by cluster]")
print(top_status)

print("\n[Top fashion_news_frequency by cluster]")
print(top_news)

print("\n[Top preferred_channel by cluster]")
print(top_channel)

print("\n[Top age_group by cluster]")
print(top_age_group)

# 군집 요약표 만들기
summary = (
    df.groupby("gmm_cluster")
      .agg(
          cluster_size=("customer_id", "count"),
          avg_age=("age", "mean"),
          avg_transactions=("Transactions", "mean")
      )
      .round(2)
      .reset_index()
)

summary = summary.merge(
    top_status[["gmm_cluster", "club_member_status"]],
    on="gmm_cluster",
    how="left"
).merge(
    top_news[["gmm_cluster", "fashion_news_frequency"]],
    on="gmm_cluster",
    how="left"
).merge(
    top_channel[["gmm_cluster", "preferred_channel"]],
    on="gmm_cluster",
    how="left"
).merge(
    top_age_group[["gmm_cluster", "age_group"]],
    on="gmm_cluster",
    how="left"
)

summary = summary.rename(columns={
    "club_member_status": "dominant_member_status",
    "fashion_news_frequency": "dominant_fashion_news",
    "preferred_channel": "dominant_channel",
    "age_group": "dominant_age_group"
})

print("\n[GMM cluster summary]")
print(summary)

# ---------------------------------------------------------
# 11. PCA Visualization
# ---------------------------------------------------------
# 샘플링 (대용량 데이터 시 시각화 부담 완화)
sample_n = min(50000, len(df))
plot_idx = np.random.RandomState(42).choice(len(df), size=sample_n, replace=False)
X_sample = X[plot_idx]
labels_sample = df["gmm_cluster"].iloc[plot_idx].values

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sample)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels_sample,
    s=8,
    alpha=0.7
)
plt.title("GMM Customer Segmentation (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 12. Simple Cluster Interpretation Draft

# ---------------------------------------------------------
print("\n[Interpretation guide]")
print("""
- 높은 Transactions + Target customer + ACTIVE:
  핵심 고객군으로 해석 가능

- preferred_channel == online 비중 높음:
  온라인 친화 고객군

- PRE-CREATE 비중 높고 Transactions 낮음:
  잠재 고객군

- fashion_news_frequency == Monthly / Regularly:
  패션 관심도가 높은 고객군

- LEFT CLUB:
  이탈/휴면 고객군
""")